In [ ]:
"""Runs a fixed HVAC control policy.
"""

import sys
sys.path.append('/kaggle/input/competitions/neurips-2026-smart-buildings/kaggle_evaluation')

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server

COMPETITION_DATA_FOLDER = '/kaggle/input/competitions/neurips-2026-smart-buildings/'


class TransferPolicy(smart_buildings_inference_server.Policy):
    """The continuous action space is normalized about zero, so zeros are the midpoint of each
    setpoint's range rather than an absence of action. Actions are constant across the episode,
    which is why the observation is unused.
    """

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        # Built once per scenario rather than per step: the spec is fixed for the episode. A real
        # policy would use this handshake to size its model against the building it just received.
        zone_count = len(building_meta_data['zone_temp_indices'])
        print(f'Controlling a building with {zone_count} zones', flush=True)

        continuous = action_spec['continuous_action']
        discrete = action_spec['discrete_action']
        # Clipped into the space's own bounds so this stays a valid submission even for a building
        # whose setpoint ranges do not straddle zero. The gateway rejects out of range actions.
        self.action = {
            'continuous_action': np.clip(np.zeros(continuous.shape), continuous.low, continuous.high).astype(np.float32),
            'discrete_action': np.clip(np.ones(discrete.shape), discrete.low, discrete.high).astype(np.int32),
        }

    def get_action(self, obs: np.ndarray) -> dict[str, np.ndarray]:
        return self.action


smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run(
    competition_data_folder=COMPETITION_DATA_FOLDER,
)